# Existing adapter: operator normalization and one correction
**Inference only. No retraining.** Uses your selected epoch-2 1.5B adapter and corrected selective-facts router.

Prepare two inputs: **selective_facts_correction_source.zip** (new source bundle) and your completed **selective_facts_1_5b.zip** (training output containing the adapter). No prior evaluation ZIP, original adapter or GitHub push is needed.

Three arms share one fresh initial model answer per task: **control**, **operator normalization**, and **normalization plus at most one correction**. Only whitespace around recognized operators is normalized. Schema/input-consistency checks trigger correction; gold contracts and reference outputs never enter that decision or feedback. Remaining invalid/inconsistent corrected answers are withheld, with no third call or reference-based fallback.

The 100 open-development cases are all reevaluated, including existing passes, with a three-case smoke first. Results record every raw reply, normalization, diagnostic, correction, environment and semantic score. Historical 85/100 is context; the fresh control is the paired comparison. Passing input checks does not prove the contract matches the user's intent.

Download **selective_facts_correction_eval.zip** and this executed notebook. Inspect gains, regressions and wrong-output acceptance before any Stage 2 integration.

**Colab:** enable a GPU (T4 or better, at least 12 GiB), put both ZIPs in `MyDrive/igt_scopefix`, then Run all. Results are copied to a timestamped Drive folder.


In [1]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
# Same Drive input folder as your successfully executed scope-fix notebook.
DRIVE_FOLDER='/content/drive/MyDrive/igt_scopefix' # change only if you stored the ZIPs elsewhere
from google.colab import drive
drive.mount('/content/drive')
INPUT=Path(DRIVE_FOLDER)
assert INPUT.is_dir(),f'Create {INPUT} in Google Drive and put the two ZIP files in it.'
for name in ['selective_facts_correction_source.zip','selective_facts_1_5b.zip']:
    found=list(INPUT.rglob(name))
    print(('found   ' if found else 'MISSING ')+name,*found)
print('The correction source ZIP is new. selective_facts_scopefix_source.zip is the previous workflow and cannot replace it.')
WORK=Path('/content/selective_correction_work')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('correction_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=INPUT/('results_'+RUN_ID)
print('Run folder:',RUN_DIR,'Results will be copied to:',DRIVE_RESULTS)


Mounted at /content/drive
found   selective_facts_correction_source.zip /content/drive/MyDrive/igt_scopefix/selective_facts_correction_source.zip
found   selective_facts_1_5b.zip /content/drive/MyDrive/igt_scopefix/selective_facts_1_5b.zip
The correction source ZIP is new. selective_facts_scopefix_source.zip is the previous workflow and cannot replace it.
Run folder: /content/selective_correction_work/correction_20261114_174227_474430 Results will be copied to: /content/drive/MyDrive/igt_scopefix/results_20261114_174227_474430


In [2]:
MANIFEST_SHA='821b007d455954aacb50a22bbd9b038628e52f279a90ea5e9dfe75289445d814'
MANIFEST_NAME='manifest.json'
ROOT=WORK/('selective_facts_correction_source_'+MANIFEST_SHA[:12])
def write_source(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong correction source ZIP version'
    manifest=json.loads(raw)
    assert manifest['experiment']=='selective-facts-correction1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source: {name}'
        target=ROOT/relative;target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():assert target.read_bytes()==data,f'Changed working source: {name}'
        else:target.write_bytes(data)
    print('Verified correction source:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob(MANIFEST_NAME):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        write_source(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('selective_facts_correction_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='selective_facts_correction_source/';name=prefix+MANIFEST_NAME
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            write_source(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place selective_facts_correction_source.zip in the input folder.'
PACKAGES=WORK/'correction_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


Verified correction source: 117 files 821b007d455954aacb50a22bbd9b038628e52f279a90ea5e9dfe75289445d814


In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
# Use the host Torch/CUDA build. Record actual environment inside every full result ZIP.
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(), 'Enable a GPU'; assert torch.cuda.get_device_properties(0).total_memory/2**30 >= 12, 'At least 12 GiB GPU memory required'; print(torch.__version__,torch.cuda.get_device_name(0))"],cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts_correction.preflight'],cwd=ROOT,env=ENV,check=True)
path_file=RUN_DIR/'adapter_path.txt'
subprocess.run([sys.executable,'-m','finetune.selective_facts_scopefix.adapter','--input',str(INPUT),'--work',str(WORK),'--out',str(path_file)],cwd=ROOT,env=ENV,check=True)
ADAPTER=Path(path_file.read_text());print('Verified existing epoch-2 adapter:',ADAPTER)


Verified existing epoch-2 adapter: /content/selective_correction_work/scopefix_existing_adapter


In [4]:
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for path in RUN_DIR.iterdir():
            if path.is_file() and path.suffix in ('.zip','.log','.json','.txt'):shutil.copy2(path,DRIVE_RESULTS/path.name)
        print('Copied results to Drive:',DRIVE_RESULTS)
def execute(smoke):
    out=RUN_DIR/('selective_facts_correction_smoke' if smoke else 'selective_facts_correction_eval')
    assert not out.exists(),'Use a fresh run folder; previous results are never overwritten.'
    command=[sys.executable,'-m','finetune.selective_facts_correction.evaluate','--adapter',str(ADAPTER),'--out',str(out),'--packages',str(PACKAGES)]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(out.name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Job failed: inspect {log} and the partial ZIP.'
    assert json.loads((out/'completion.json').read_text())['complete']
    return out
SMOKE=execute(True)
print('Three-case smoke finished. Next cell runs the full fresh paired evaluation.')



Loading weights: 100%|██████████| 338/338 [00:08<00:00, 39.10it/s]
1/3 warehouse_filter_0: {"control": false, "normalized": true, "corrected": true} correction=False
2/3 warehouse_edit_1: {"control": false, "normalized": false, "corrected": false} correction=True
3/3 warehouse_dedup_0: {"control": true, "normalized": true, "corrected": true} correction=False
{
  "current": {
    "control": {
      "synthetic": {
        "tasks": 3,
        "correct_accepted": 1,
        "wrong_accepted": 0,
        "joint_success": 1,
        "invalid": 1,
        "tokens": 4763
      },
      "original": {
        "tasks": 0,
        "correct_accepted": 0,
        "wrong_accepted": 0,
        "joint_success": 0,
        "invalid": 0,
        "tokens": 0
      }
    },
    "normalized": {
      "synthetic": {
        "tasks": 3,
        "correct_accepted": 2,
        "wrong_accepted": 0,
        "joint_success": 2,
        "invalid": 0,
        "tokens": 4763
      },
      "original": {
        "task

In [5]:
OUT=execute(False)
print((OUT/'comparison.json').read_text())
print('Download:',OUT.with_suffix('.zip'))
if DRIVE_RESULTS is not None:print('Drive copy:',DRIVE_RESULTS/'selective_facts_correction_eval.zip')
from IPython.display import display,FileLink
display(FileLink(str(OUT.with_suffix('.zip'))))
print('Download the executed notebook too. No training was performed.')



Loading weights: 100%|██████████| 338/338 [00:02<00:00, 116.50it/s]
1/100 warehouse_filter_0: {"control": false, "normalized": true, "corrected": true} correction=False
2/100 warehouse_filter_1: {"control": true, "normalized": true, "corrected": true} correction=False
3/100 warehouse_filter_2: {"control": true, "normalized": true, "corrected": true} correction=False
4/100 warehouse_filter_3: {"control": true, "normalized": true, "corrected": true} correction=False
5/100 warehouse_dedup_0: {"control": true, "normalized": true, "corrected": true} correction=False
6/100 warehouse_dedup_1: {"control": true, "normalized": true, "corrected": true} correction=False
7/100 warehouse_dedup_2: {"control": true, "normalized": true, "corrected": true} correction=False
8/100 warehouse_dedup_3: {"control": true, "normalized": true, "corrected": true} correction=False
9/100 warehouse_fill_0: {"control": true, "normalized": true, "corrected": true} correction=False
10/100 warehouse_fill_1: {"control":

/content/selective_correction_work/correction_20261114_174227_474430/selective_facts_correction_eval.zip

Download the executed notebook too. No training was performed.
